
## Folder 07 — File 03: Gold Customer Lifetime Value (CLV):


This notebook creates** customer-level analytics** using **Silver customers, orders** and **payments**.

It calculates:

- Total orders per customer.
- Total order value.
- Successful payment amount.
- Refunded payment amount.
- Net collected payment amount.
- Average order value.
- Customer purchase frequency.
- First and latest order dates.
- Customer lifetime value (CLV).
- Customer segmentation.

**Important distinction:**

Order value and collected payment are different metrics. This notebook keeps them separate. It does not treat failed or pending payments as collected revenue.

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

CUSTOMERS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_customers"
)

ORDERS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_orders"
)

PAYMENTS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_payments"
)

GOLD_CUSTOMER_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_customer_lifetime_value"
)

GOLD_SEGMENT_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_customer_segments"
)

RUN_ID = spark.sql(
    "SELECT uuid() AS run_id"
).first()["run_id"]

RUN_TIMESTAMP = datetime.now()

print("Run ID:", RUN_ID)


## 3. Validate source tables

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}"
)

SOURCE_TABLES = [
    CUSTOMERS_TABLE,
    ORDERS_TABLE,
    PAYMENTS_TABLE
]

for table_name in SOURCE_TABLES:

    if not spark.catalog.tableExists(table_name):
        raise Exception(
            f"Missing source table: {table_name}"
        )

print("All source tables are available.")


## 4. Load Silver datasets

customers_df = spark.table(CUSTOMERS_TABLE)

orders_df = spark.table(ORDERS_TABLE)

payments_df = spark.table(PAYMENTS_TABLE)

print("Customers:", customers_df.count())
print("Orders:", orders_df.count())
print("Payments:", payments_df.count())


## 5. Validate required columns

required_customer_columns = [
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "created_at"
]

required_order_columns = [
    "order_id",
    "customer_id",
    "order_timestamp",
    "order_status",
    "order_total",
    "currency"
]

required_payment_columns = [
    "payment_id",
    "order_id",
    "payment_status",
    "amount",
    "payment_timestamp"
]

for column_name in required_customer_columns:

    if column_name not in customers_df.columns:
        raise Exception(
            f"Missing customer column: {column_name}"
        )

for column_name in required_order_columns:

    if column_name not in orders_df.columns:
        raise Exception(
            f"Missing order column: {column_name}"
        )

for column_name in required_payment_columns:

    if column_name not in payments_df.columns:
        raise Exception(
            f"Missing payment column: {column_name}"
        )


## 6. Clean customer data

customers_clean_df = (
    customers_df
    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id"))
    )
    .withColumn(
        "first_name",
        F.trim(F.col("first_name"))
    )
    .withColumn(
        "last_name",
        F.trim(F.col("last_name"))
    )
    .withColumn(
        "email",
        F.lower(F.trim(F.col("email")))
    )
    .withColumn(
        "created_at",
        F.to_date("created_at")
    )
    .filter(
        F.col("customer_id").isNotNull()
        & (F.col("customer_id") != "")
    )
)


## 7. Deduplicate customer records

if "_ingestion_timestamp" in customers_clean_df.columns:

    customer_window = (
        Window
        .partitionBy("customer_id")
        .orderBy(
            F.col("_ingestion_timestamp").desc_nulls_last()
        )
    )

else:

    customer_window = (
        Window
        .partitionBy("customer_id")
        .orderBy(F.col("customer_id"))
    )

customers_deduplicated_df = (
    customers_clean_df
    .withColumn(
        "_row_number",
        F.row_number().over(customer_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)


## 8. Prepare order data

orders_clean_df = (
    orders_df
    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id"))
    )
    .withColumn(
        "order_status",
        F.upper(F.trim(F.col("order_status")))
    )
    .withColumn(
        "order_timestamp",
        F.to_timestamp("order_timestamp")
    )
    .withColumn(
        "order_date",
        F.to_date("order_timestamp")
    )
    .withColumn(
        "order_total",
        F.col("order_total").cast("decimal(18,2)")
    )
    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency")))
    )
    .filter(
        F.col("order_id").isNotNull()
        & F.col("customer_id").isNotNull()
        & F.col("order_timestamp").isNotNull()
        & F.col("order_total").isNotNull()
        & (F.col("order_total") >= 0)
        & F.col("currency").isNotNull()
    )
)


## 9. Deduplicate orders

if "_ingestion_timestamp" in orders_clean_df.columns:

    order_window = (
        Window
        .partitionBy("order_id")
        .orderBy(
            F.col("_ingestion_timestamp").desc_nulls_last()
        )
    )

else:

    order_window = (
        Window
        .partitionBy("order_id")
        .orderBy(
            F.col("order_timestamp").desc()
        )
    )

orders_deduplicated_df = (
    orders_clean_df
    .withColumn(
        "_row_number",
        F.row_number().over(order_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)


 ## 10. Aggregate customer order metrics

# Cancelled orders are excluded from purchase value and order frequency.
# All order statuses are retained in the separate operational count.


valid_customer_orders_df = (
    orders_deduplicated_df
    .filter(
        F.col("order_status") != "CANCELLED"
    )
)

customer_order_metrics_df = (
    valid_customer_orders_df
    .groupBy(
        "customer_id",
        "currency"
    )
    .agg(

        F.countDistinct("order_id").alias(
            "total_orders"
        ),

        F.sum("order_total").cast("decimal(18,2)").alias(
            "total_order_value"
        ),

        F.avg("order_total").cast("decimal(18,2)").alias(
            "average_order_value"
        ),

        F.min("order_date").alias(
            "first_order_date"
        ),

        F.max("order_date").alias(
            "last_order_date"
        ),

        F.sum(
            F.when(
                F.col("order_status") == "DELIVERED",
                1
            ).otherwise(0)
        ).alias(
            "delivered_orders"
        ),

        F.sum(
            F.when(
                F.col("order_status") == "SHIPPED",
                1
            ).otherwise(0)
        ).alias(
            "shipped_orders"
        ),

        F.sum(
            F.when(
                F.col("order_status") == "PROCESSING",
                1
            ).otherwise(0)
        ).alias(
            "processing_orders"
        )
    )
)


## 11. Prepare payment data

payments_clean_df = (
    payments_df
    .withColumn(
        "payment_status",
        F.upper(F.trim(F.col("payment_status")))
    )
    .withColumn(
        "amount",
        F.col("amount").cast("decimal(18,2)")
    )
    .withColumn(
        "payment_timestamp",
        F.to_timestamp("payment_timestamp")
    )
    .filter(
        F.col("payment_id").isNotNull()
        & F.col("order_id").isNotNull()
        & F.col("amount").isNotNull()
        & (F.col("amount") >= 0)
    )
)


## 12. Deduplicate payment records

if "_ingestion_timestamp" in payments_clean_df.columns:

    payment_window = (
        Window
        .partitionBy("payment_id")
        .orderBy(
            F.col("_ingestion_timestamp").desc_nulls_last()
        )
    )

else:

    payment_window = (
        Window
        .partitionBy("payment_id")
        .orderBy(
            F.col("payment_timestamp").desc_nulls_last()
        )
    )

payments_deduplicated_df = (
    payments_clean_df
    .withColumn(
        "_row_number",
        F.row_number().over(payment_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)


## 13. Aggregate payments by order

# Multiple payment records can belong to one order.
# Aggregate them before joining to orders to prevent row multiplication.

payment_order_metrics_df = (
    payments_deduplicated_df
    .groupBy("order_id")
    .agg(

        F.sum(
            F.when(
                F.col("payment_status") == "SUCCESS",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "successful_payment_amount"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "REFUNDED",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "refunded_payment_amount"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "PENDING",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "pending_payment_amount"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "FAILED",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "failed_payment_amount"
        ),

        F.countDistinct("payment_id").alias(
            "payment_attempt_count"
        )
    )
)


## 14. Join payments to orders

# First aggregate payment records at order level.

# Then join the one-row-per-order result to customer orders.

orders_with_payments_df = (
    valid_customer_orders_df.alias("orders")
    .join(
        payment_order_metrics_df.alias("payments"),
        F.col("orders.order_id")
        == F.col("payments.order_id"),
        "left"
    )
    .select(
        F.col("orders.order_id"),
        F.col("orders.customer_id"),
        F.col("orders.currency"),
        F.col("orders.order_total"),
        F.col("orders.order_status"),

        F.coalesce(
            F.col("payments.successful_payment_amount"),
            F.lit(0).cast("decimal(18,2)")
        ).alias("successful_payment_amount"),

        F.coalesce(
            F.col("payments.refunded_payment_amount"),
            F.lit(0).cast("decimal(18,2)")
        ).alias("refunded_payment_amount"),

        F.coalesce(
            F.col("payments.pending_payment_amount"),
            F.lit(0).cast("decimal(18,2)")
        ).alias("pending_payment_amount"),

        F.coalesce(
            F.col("payments.failed_payment_amount"),
            F.lit(0).cast("decimal(18,2)")
        ).alias("failed_payment_amount"),

        F.coalesce(
            F.col("payments.payment_attempt_count"),
            F.lit(0)
        ).alias("payment_attempt_count")
    )
    .withColumn(
        "net_collected_payment",
        (
            F.col("successful_payment_amount")
            - F.col("refunded_payment_amount")
        ).cast("decimal(18,2)")
    )
)


## 15. Aggregate customer payment metrics

customer_payment_metrics_df = (
    orders_with_payments_df
    .groupBy(
        "customer_id",
        "currency"
    )
    .agg(

        F.sum("successful_payment_amount").cast(
            "decimal(18,2)"
        ).alias(
            "total_successful_payments"
        ),

        F.sum("refunded_payment_amount").cast(
            "decimal(18,2)"
        ).alias(
            "total_refunded_payments"
        ),

        F.sum("net_collected_payment").cast(
            "decimal(18,2)"
        ).alias(
            "net_collected_payment"
        ),

        F.sum("pending_payment_amount").cast(
            "decimal(18,2)"
        ).alias(
            "total_pending_payments"
        ),

        F.sum("failed_payment_amount").cast(
            "decimal(18,2)"
        ).alias(
            "total_failed_payments"
        ),

        F.sum("payment_attempt_count").alias(
            "total_payment_attempts"
        )
    )
)


## 16. Join customer master with order metrics

# Customers without purchases are retained.

customer_base_df = (
    customers_deduplicated_df.alias("customers")
    .join(
        customer_order_metrics_df.alias("orders"),
        F.col("customers.customer_id")
        == F.col("orders.customer_id"),
        "left"
    )
    .select(
        F.col("customers.customer_id"),
        F.col("customers.first_name"),
        F.col("customers.last_name"),
        F.col("customers.email"),
        F.col("customers.city"),
        F.col("customers.state"),
        F.col("customers.created_at"),

        F.col("orders.currency"),
        F.col("orders.total_orders"),
        F.col("orders.total_order_value"),
        F.col("orders.average_order_value"),
        F.col("orders.first_order_date"),
        F.col("orders.last_order_date"),
        F.col("orders.delivered_orders"),
        F.col("orders.shipped_orders"),
        F.col("orders.processing_orders")
    )
)

## 17. Join customer payment metrics

customer_with_payments_df = (
    customer_base_df.alias("customers")
    .join(
        customer_payment_metrics_df.alias("payments"),
        (
            F.col("customers.customer_id")
            == F.col("payments.customer_id")
        )
        & (
            F.col("customers.currency")
            == F.col("payments.currency")
        ),
        "left"
    )
    .select(
        "customers.*",

        F.col("payments.total_successful_payments"),
        F.col("payments.total_refunded_payments"),
        F.col("payments.net_collected_payment"),
        F.col("payments.total_pending_payments"),
        F.col("payments.total_failed_payments"),
        F.col("payments.total_payment_attempts")
    )
)

## 18. Fill missing metrics

metric_columns = [
    "total_orders",
    "total_order_value",
    "average_order_value",
    "delivered_orders",
    "shipped_orders",
    "processing_orders",
    "total_successful_payments",
    "total_refunded_payments",
    "net_collected_payment",
    "total_pending_payments",
    "total_failed_payments",
    "total_payment_attempts"
]

customer_with_payments_df = customer_with_payments_df.fillna(
    0,
    subset=metric_columns
)


## 19. Calculate customer purchase frequency

customer_with_payments_df = (
    customer_with_payments_df
    .withColumn(
        "purchase_frequency",
        F.when(
            F.col("total_orders") > 0,
            F.round(
                F.col("total_orders")
                / F.greatest(
                    F.datediff(
                        F.col("last_order_date"),
                        F.col("first_order_date")
                    ),
                    F.lit(1)
                ),
                4
            )
        ).otherwise(F.lit(0))
    )
)


## 20. Calculate customer lifetime value

# Historical CLV here is defined as cumulative net collected payment.
# It is not a prediction of future customer spending.

customer_with_payments_df = (
    customer_with_payments_df
    .withColumn(
        "historical_customer_lifetime_value",
        F.col("net_collected_payment").cast("decimal(18,2)")
    )
    .withColumn(
        "estimated_order_value_lifetime",
        F.col("total_order_value").cast("decimal(18,2)")
    )
)


## 21. Assign customer segments

# These are simple rule-based segments.

# They are not machine-learning predictions.


customer_with_payments_df = (
    customer_with_payments_df
    .withColumn(
        "customer_segment",
        F.when(
            F.col("total_orders") == 0,
            "NO_PURCHASE"
        )
        .when(
            F.col("historical_customer_lifetime_value") >= 10000,
            "HIGH_VALUE"
        )
        .when(
            F.col("historical_customer_lifetime_value") >= 3000,
            "MEDIUM_VALUE"
        )
        .otherwise("LOW_VALUE")
    )
)


## 22. Rank customers by lifetime value

customer_rank_window = (
    Window
    .partitionBy("currency")
    .orderBy(
        F.col("historical_customer_lifetime_value").desc()
    )
)

customer_with_payments_df = (
    customer_with_payments_df
    .withColumn(
        "customer_value_rank",
        F.when(
            F.col("currency").isNotNull(),
            F.dense_rank().over(customer_rank_window)
        )
    )
)


## 23. Add audit columns

customer_final_df = (
    customer_with_payments_df
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .withColumn(
        "created_at_gold",
        F.lit(RUN_TIMESTAMP)
    )
)


## 24. Create customer segment summary

customer_segment_df = (
    customer_final_df
    .groupBy(
        "customer_segment",
        "currency"
    )
    .agg(

        F.countDistinct("customer_id").alias(
            "total_customers"
        ),

        F.sum("total_orders").alias(
            "total_orders"
        ),

        F.sum("total_order_value").cast(
            "decimal(18,2)"
        ).alias(
            "total_order_value"
        ),

        F.sum("net_collected_payment").cast(
            "decimal(18,2)"
        ).alias(
            "net_collected_payment"
        ),

        F.avg(
            "historical_customer_lifetime_value"
        ).cast("decimal(18,2)").alias(
            "average_customer_lifetime_value"
        )
    )
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)


## 25. Display top 10 customers

display(
    customer_final_df
    .filter(
        F.col("currency").isNotNull()
    )
    .orderBy(
        F.col("historical_customer_lifetime_value").desc()
    )
    .limit(10)
)


## 26. Display customer segments

display(
    customer_segment_df.orderBy(
        F.col("total_customers").desc()
    )
)


## 27. Save Gold customer lifetime value table

(
    customer_final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_CUSTOMER_TABLE)
)

print(
    "Gold customer lifetime value table created:",
    GOLD_CUSTOMER_TABLE
)


## 28. Save customer segment table

(
    customer_segment_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_SEGMENT_TABLE)
)

print(
    "Gold customer segment table created:",
    GOLD_SEGMENT_TABLE
)


## 29. Final validation

gold_customer_df = spark.table(GOLD_CUSTOMER_TABLE)

gold_segment_df = spark.table(GOLD_SEGMENT_TABLE)

print("Gold customer records:", gold_customer_df.count())

print("Gold customer segment records:", gold_segment_df.count())

display(
    gold_customer_df
    .groupBy("customer_segment")
    .agg(
        F.count("*").alias("customer_count"),
        F.sum("historical_customer_lifetime_value").alias(
            "total_customer_value"
        )
    )
)

print("=" * 65)
print("GOLD CUSTOMER LIFETIME VALUE COMPLETED")
print("=" * 65)

print("Customer table:", GOLD_CUSTOMER_TABLE)
print("Segment table:", GOLD_SEGMENT_TABLE)


### Top 10 customers

In [0]:
%sql

SELECT
    customer_id,
    first_name,
    last_name,
    total_orders,
    total_order_value,
    net_collected_payment,
    historical_customer_lifetime_value,
    customer_value_rank
FROM workspace.ecommerce_gold.gold_customer_lifetime_value
WHERE currency IS NOT NULL
ORDER BY historical_customer_lifetime_value DESC
LIMIT 10;


### Customer segment distribution

In [0]:
%sql

SELECT
    customer_segment,
    currency,
    total_customers,
    total_orders,
    total_order_value,
    net_collected_payment,
    average_customer_lifetime_value
FROM workspace.ecommerce_gold.gold_customer_segments
ORDER BY total_customers DESC;


### Customers with no purchases

In [0]:
%sql

SELECT
    customer_id,
    first_name,
    last_name,
    email,
    city,
    state
FROM workspace.ecommerce_gold.gold_customer_lifetime_value
WHERE customer_segment = 'NO_PURCHASE';


### High-value customers

In [0]:
%sql

SELECT
    customer_id,
    first_name,
    last_name,
    total_orders,
    historical_customer_lifetime_value,
    last_order_date
FROM workspace.ecommerce_gold.gold_customer_lifetime_value
WHERE customer_segment = 'HIGH_VALUE'
ORDER BY historical_customer_lifetime_value DESC;

In [0]:
%sql

DESCRIBE TABLE workspace.ecommerce_gold.gold_payment_metrics;

In [0]:
%sql

DESCRIBE TABLE workspace.ecommerce_gold.gold_executive_kpis;
DESCRIBE TABLE workspace.ecommerce_gold.gold_daily_sales;
DESCRIBE TABLE workspace.ecommerce_gold.gold_channel_daily_sales;
DESCRIBE TABLE workspace.ecommerce_gold.gold_product_performance;
DESCRIBE TABLE workspace.ecommerce_gold.gold_category_performance;
DESCRIBE TABLE workspace.ecommerce_gold.gold_customer_lifetime_value;
DESCRIBE TABLE workspace.ecommerce_gold.gold_customer_segments;
DESCRIBE TABLE workspace.ecommerce_gold.gold_payment_method_performance;
DESCRIBE TABLE workspace.ecommerce_gold.gold_shipping_metrics;
DESCRIBE TABLE workspace.ecommerce_gold.gold_inventory_status;
DESCRIBE TABLE workspace.ecommerce_gold.gold_warehouse_inventory;
DESCRIBE TABLE workspace.ecommerce_gold.gold_inventory_snapshot_trends;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_customer_lifetime_value;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_customer_segments;
DESCRIBE TABLE workspace.ecommerce_gold.gold_category_performance;